# 8.4. Multi-Branch Networks (GoogLeNet)
D2L의 Multi-Branch Networks (GoogLeNet)장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. GoogLeNet이 등장한 이유

AlexNet, VGG에서는 대체로 Convolution layer를 쌓았다. 

그런데 어떤 크기의 kernel을 사용하는 것이 가장 좋은지에 대한 문제가 생긴다.

작은 패턴은 1x1, 3x3이 잘 볼 수 있고, 더 넓은 영역의 패턴은 5x5 같은 큰 kernel이 유리할 수 있다.

GoogLeNet은 하나만 고르지 말고 여러 convolution을 동시에 사용하면 어떨까? 라는 아이디어를 냈다. 이게 Inception block이다.

## 2. GoogLeNet의 아이디어

기존 CNN은 한 층에서 하나의 종류의 convolution을 수행한다. 

하지만 GoogLeNet은 하나의 입력을 여러 경로로 보낸다.
```text
                 ┌─ 1×1 Conv ──────────────┐
                 │                         │
입력 ────────────┼─ 1×1 Conv → 3×3 Conv  ──┤
                 │                         ├─ Concat → 출력
                 ├─ 1×1 Conv → 5×5 Conv -──┤
                 │                         │
                 └─ MaxPool → 1×1 Conv -───┘
```
하나의 feature map을 서로 다른 방식으로 분석한 뒤 결과를 합친다. 이 구조가 Inception block이라고 한다.

## 3. 왜 여러 크기의 kernel을 사용할까?

kernel 크기가 다르면 볼 수 있는 공간적 범위도 달라진다.

예를 들어 이미지에 고양이가 있다고 했을때

```text
작은 영역
-> 털의 경계
-> 작은 선
-> 작은 무늬

중간 영역
-> 귀
-> 눈
-> 코

큰 영역
-> 얼굴 형태
-> 여러 특징의 조합
```

각각 서로 다른 종류의 정보를 추출할 수 있다. 어떤 kernel이 가장 좋은지 선택하는 대신 여러 kernel을 동시에 사용한다.

## 4. Inception Block의 네가지 Branch

Inception Block에는 크게 네 개의 branch가 있다.

### Branch 1
입력
 ↓
1×1 Conv

주로 채널 정보를 조합한다.

### Branch 2
입력
 ↓
1×1 Conv
 ↓
3×3 Conv

중간 정도 크기의 공간적 특징을 추출한다.

### Branch 3
입력
 ↓
1×1 Conv
 ↓
5×5 Conv

더 넓은 영역의 특징을 추출한다.

### Branch 4
입력
 ↓
3×3 MaxPool
 ↓
1×1 Conv

Pooling으로 얻은 특징도 함께 사용한다. 

입력을 네 가지 관점에서 처리하는 것이다.

## 5. 여기서 1x1 Conv는 왜 이렇게 많나?

GoogLeNet에서 매우 중요한 부분이다. 1×1 Conv는 단순히 의미 없는 작은 convolution이 아니다.

공간 크기 H × W는 그대로 유지하면서 채널 수를 변경할 수 있다.

예를 들어서 입력이 [batch, 192, 28, 28] 이렇다고 하자.

바로 5×5 Conv를 수행하면 계산량이 매우 커진다.

대신 먼저

```text
1×1 Conv

192 channels -> 32 channels
```

처럼 채널을 줄인다.

그 다음 5×5 Conv 를 수행한다.

```text
192 channels -> 1×1 Conv -> 32 channels -> 5×5 Conv
```

이것을 `bottleneck` 구조라고 볼 수 있다.

핵심 목적은 이렇다.

- 채널 수 감소
- 파라미터 감소
- 연산량 감소
- 필요한 특징 조합

## 6. 각 Brach의 결과는 어떻게 합치는가?

각 brach는 padding을 적절하게 사용해 H, W 크기가 동일하게 만든다. 

예를 들어서 네 brach 결과가 다음과 같다고 했을때

```text
Branch 1 : [N, 64, H, W] 
Branch 2 : [N, 128, H, W] 
Branch 3 : [N, 32, H, W] 
Branch 4 : [N, 32, H, W]
```

이것들을 채널 방향으로 연결(concatenate)한다.

따라서 출력은 [N, 256, H, W]가 된다. (64 + 128 + 32 + 32 = 256)

```py
out = torch.cat( 
    [branch1, branch2, branch3, branch4], 
    dim=1 # [N, C, H, W] 에서 채널 C방향으로 붙인다는 뜻
)
```

## 7. Inception Block 구현

In [3]:
class InceptionBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        c1,
        c2_reduce,
        c2,
        c3_reduce,
        c3,
        c4
    ):
        super().__init__()

        # Branch 1
        self.branch1 = nn.Conv2d(
            in_channels,
            c1,
            kernel_size=1
        )

        # Branch 2
        self.branch2_reduce = nn.Conv2d(
            in_channels,
            c2_reduce,
            kernel_size=1
        )

        self.branch2 = nn.Conv2d(
            c2_reduce,
            c2,
            kernel_size=3,
            padding=1
        )

        # Branch 3
        self.branch3_reduce = nn.Conv2d(
            in_channels,
            c3_reduce,
            kernel_size=1
        )

        self.branch3 = nn.Conv2d(
            c3_reduce,
            c3,
            kernel_size=5,
            padding=2
        )

        # Branch 4
        self.pool = nn.MaxPool2d(
            kernel_size=3,
            stride=1,
            padding=1
        )

        self.branch4 = nn.Conv2d(
            in_channels,
            c4,
            kernel_size=1
        )

    def forward(self, x):

        b1 = F.relu(self.branch1(x))

        b2 = F.relu(self.branch2_reduce(x))
        b2 = F.relu(self.branch2(b2))

        b3 = F.relu(self.branch3_reduce(x))
        b3 = F.relu(self.branch3(b3))

        b4 = self.pool(x)
        b4 = F.relu(self.branch4(b4))

        return torch.cat(
            [b1, b2, b3, b4],
            dim=1
        )

## 8. GoogLeNet 전체 구조

GoogLeNet은 Inception Block 하나만 사용하는 모델이 아니다. 전체적으로 본다면 이렇다.

```text
입력 이미지
    ↓
초기 Conv + Pool
    ↓
Conv + Pool
    ↓
Inception × 2
    ↓
Pooling
    ↓
Inception × 5
    ↓
Pooling
    ↓
Inception × 2
    ↓
Global Average Pooling
    ↓
Linear
    ↓
Class Prediction
```

총 9개의 Inception block이 사용된다.

초기 부분은 일반 CNN처럼 기본적인 저수준 특징을 추출하고, 중간 이후부터 Inception block을 이용해 다양한 크기의 특징을 동시에 추출한다.

## 9. GoogLeNet의 Shape 변화

Fashion-MNIST 이미지를 96 x 96으로 확대해서 사용한다. 

shape 변화
```text
입력
[1, 1, 96, 96]
↓
[1, 64, 24, 24]
↓
[1, 192, 12, 12]
↓
[1, 480, 6, 6]
↓
[1, 832, 3, 3]
↓
Global Average Pooling
↓
[1, 1024]
↓
Linear
↓
[1, 10]
```

전 CNN들과 동일한 특성이 있다. H,W는 계속 작아지지만 채널 수는 증가한다.

> 이미지의 위치 정보는 점점 압축하고, 추출된 특징의 종류는 점점 증가시킨다.

## 10. Global Average Pooling

GoogLeNet 마지막에서는 NiN 마지막에 본 Global Average Pooling을 사용한다.

예를 들어 [N, 1024, 3, 3] 이 있다면 각 채널의 3×3 값을 평균낸다.

[N, 1024, 3, 3] -> [N, 1024, 1, 1] -> Flatten -> [N, 1024]

이렇게 하면 거대한 Fully Connected Layer를 사용할 필요가 없다. 파라미터 수를 크게 줄일 수 있다.

## 11. GoogLeNet이 중요한 이유

GoogLeNet 이전에는 CNN들은 주로 직렬로 연결되었다.

GoogLeNet은 이를 여러 branch를 가진 네트워크로 확장했다. 그리고 각 branch가 서로 다른 크기의 receptive field를 사용한다.

결과적으로 작은 특징, 중간 특징, 큰 특징, Pooling 특징 동시에 얻을 수 있다.

또한 1x1 Conv를 이용한 채널 축소 덕분에 깊고 복잡한 네트워크임에도 계산량을 제어할 수 있다.

## 12. 오늘의 정리

- GoogLeNet은 2014 ImageNet에서 성공한 대표적인 현대 CNN 구조다.
- GoogLeNet의 핵심은 Inception block이다.
- Inception은 하나의 입력을 여러 branch로 동시에 보낸다.
- 1×1, 3×3, 5×5 Conv, MaxPool을 동시에 사용한다.
- 서로 다른 kernel 크기를 통해 다양한 범위의 특징을 추출한다.
- 각 branch의 결과는 채널 방향으로 concatenate한다.
- 1×1 Conv는 채널을 줄여 연산량과 파라미터 수를 감소시키는 중요한 역할을 한다.
- GoogLeNet은 총 9개의 Inception block을 사용한다.
- 깊어질수록 H, W는 감소하고 채널 수는 증가한다.
- 마지막에는 Global Average Pooling을 사용하여 거대한 Fully Connected Layer를 피한다.
- GoogLeNet의 핵심 철학은 어떤 convolution이 좋은지 하나를 고르는 대신 여러 종류를 동시에 사용하자는 것이다.